# Notebook 08 — Shoreline Change Analysis

This notebook measures shoreline positions for 2021–2024 along the 2024 transects.
It calculates change rates only where positions exist for at least two years.
Results are preliminary until the coastal NoData QA issue is resolved.

In [ ]:
import sys
from pathlib import Path
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np

cwd = Path.cwd()
if (cwd / "notebooks").exists():
    project_root = cwd
elif (cwd.parent / "notebooks").exists():
    project_root = cwd.parent
else:
    project_root = cwd

PROJECT_ROOT = project_root
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRANSECTS_PATH = (
    PROJECT_ROOT / "outputs" / "transects" / "2024" / "shoreline_transects_2024.gpkg"
)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "change"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2021, 2022, 2023, 2024]
SHORELINE_PATHS = {
    year: PROCESSED_DIR / "mombasa_test" / str(year) / f"shoreline_{year}_candidates.gpkg"
    for year in YEARS
}

print("Transects:", TRANSECTS_PATH)
print("Output:", OUTPUT_DIR)
print("Years:", YEARS)
for year in YEARS:
    print(f"  {year}: {SHORELINE_PATHS[year]}")


In [ ]:
if not TRANSECTS_PATH.exists():
    raise FileNotFoundError(f"Missing input: {TRANSECTS_PATH}")
transects = gpd.read_file(TRANSECTS_PATH)

shorelines = {}
for year in YEARS:
    path = SHORELINE_PATHS[year]
    if not path.exists():
        raise FileNotFoundError(f"Missing input for {year}: {path}")
    shorelines[year] = gpd.read_file(path)

print("Transects CRS:", transects.crs, "count:", len(transects))
print("Transects bounds:", transects.total_bounds)
for year in YEARS:
    s = shorelines[year]
    print(f"{year} shoreline CRS:", s.crs, "count:", len(s))
    print(f"{year} shoreline bounds:", s.total_bounds)

if transects.empty:
    raise ValueError("Transects are empty")
for year in YEARS:
    if shorelines[year].empty:
        raise ValueError(f"{year} shoreline candidates are empty")
    if shorelines[year].crs != transects.crs:
        raise ValueError(
            f"CRS mismatch for {year}: {shorelines[year].crs} vs transects {transects.crs}"
        )


In [ ]:
all_records = []
intersection_id = 0

for year in YEARS:
    shoreline = shorelines[year]
    shoreline_union = shoreline.geometry.unary_union
    for _, trow in transects.iterrows():
        tid = trow.get("transect_id", trow.name)
        inter = trow.geometry.intersection(shoreline_union)
        pts = []
        if inter.is_empty:
            pts = []
        elif inter.geom_type == "Point":
            pts = [inter]
        elif inter.geom_type == "MultiPoint":
            pts = list(inter.geoms)
        elif hasattr(inter, "geoms"):
            for g in inter.geoms:
                if g.geom_type == "Point":
                    pts.append(g)
        for p in pts:
            intersection_id += 1
            all_records.append(
                {
                    "year": year,
                    "transect_id": tid,
                    "intersection_id": intersection_id,
                    "geometry": p,
                }
            )

positions = gpd.GeoDataFrame(all_records, crs=transects.crs)
positions["longitude"] = positions.geometry.x
positions["latitude"] = positions.geometry.y

out_gpkg = OUTPUT_DIR / "shoreline_positions_all_years.gpkg"
positions.to_file(out_gpkg, driver="GPKG", layer="shoreline_positions")
out_csv = OUTPUT_DIR / "shoreline_positions_all_years.csv"
positions[["year", "transect_id", "intersection_id", "longitude", "latitude"]].to_csv(
    out_csv, index=False
)

print("Saved:", out_gpkg)
print("Saved:", out_csv)
print("Intersection counts per year:")
counts = positions.groupby("year").size()
for year in YEARS:
    print(f"  {year}: {int(counts.get(year, 0))}")


In [ ]:
origins = {}
for _, trow in transects.iterrows():
    tid = trow.get("transect_id", trow.name)
    coords = list(trow.geometry.coords)
    origins[tid] = coords[0]

rate_records = []
n_rates = 0
n_insufficient = 0

for tid in sorted(origins.keys()):
    sub = positions[positions["transect_id"] == tid]
    years_present = sorted(sub["year"].unique())
    if len(years_present) < 2:
        rate_records.append(
            {
                "transect_id": tid,
                "n_years": int(len(years_present)),
                "years": ",".join(str(y) for y in years_present),
                "change_rate_m_per_year": None,
                "status": "insufficient_data",
            }
        )
        n_insufficient += 1
        continue
    origin = origins[tid]
    xs = []
    ys = []
    for _, prow in sub.iterrows():
        d = float(np.hypot(prow.geometry.x - origin[0], prow.geometry.y - origin[1]))
        xs.append(int(prow["year"]))
        ys.append(d)
    slope, _ = np.polyfit(np.array(xs, dtype=float), np.array(ys, dtype=float), 1)
    rate_records.append(
        {
            "transect_id": tid,
            "n_years": int(len(years_present)),
            "years": ",".join(str(y) for y in years_present),
            "change_rate_m_per_year": float(slope),
            "status": "computed",
        }
    )
    n_rates += 1

rates = pd.DataFrame(
    rate_records,
    columns=[
        "transect_id",
        "n_years",
        "years",
        "change_rate_m_per_year",
        "status",
    ],
)
rates_path = OUTPUT_DIR / "shoreline_change_rates.csv"
rates.to_csv(rates_path, index=False)

print("Saved:", rates_path)
print(rates.head(10).to_string(index=False))
print(f"Transects with rates: {n_rates}")
print(f"Transects insufficient_data: {n_insufficient}")


In [ ]:
import folium


def rate_color(rate):
    if rate is None:
        return "gray"
    try:
        r = float(rate)
    except (TypeError, ValueError):
        return "gray"
    if r != r:
        return "gray"
    if r < -0.5:
        return "red"
    if r > 0.5:
        return "green"
    return "orange"


transects_wgs = transects.to_crs(epsg=4326)
rates_indexed = rates.set_index("transect_id")

center = [transects_wgs.total_bounds[1], transects_wgs.total_bounds[0]]
m = folium.Map(location=center, zoom_start=12)
for _, row in transects_wgs.iterrows():
    tid = row.get("transect_id", row.name)
    rate = (
        rates_indexed.loc[tid, "change_rate_m_per_year"]
        if tid in rates_indexed.index
        else None
    )
    color = rate_color(rate)
    coords = (
        [(y, x) for x, y in row.geometry.coords]
        if hasattr(row.geometry, "coords")
        else []
    )
    folium.PolyLine(coords, color=color, weight=3).add_to(m)

map_path = OUTPUT_DIR / "shoreline_change_map.html"
m.save(str(map_path))
print("Map saved:", map_path)


Change rates are preliminary and reflect measured shoreline movement only.
The unresolved coastal NoData QA issue may affect all years.
Validation in QGIS is required before interpreting these results.
Do not interpret causes of change.